# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [7]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
## Unit of Analysis



#One row represents one content item for one report date from the
#fact_content_daily_performance table.

#fact_content_daily_performance

#Time window:
#March 2026 (month = 2026-03)

#Prediction goal:
#Predict whether a content item is likely to decline in search performance.

#Excluded:
#Future information such as trend labels and future traffic metrics because
#they would cause data leakage.


In [8]:
# This cell is for CODE (numbers, a query, a check).
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS cnt
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY
    client_hash_id,
    content_hash_id,
    report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,cnt


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [15]:
### Features

#- gsc_impressions
#- gsc_clicks
#- gsc_avg_position
#- ga4_sessions
#- content_age_days

### Label

##Future decline in search performance.

### Context

#- client_hash_id
#- content_hash_id
#- report_date

### Excluded

##- trend_direction
##- trend_pct
##- future traffic values

##Reason:
##These contain future information and would leak the answer into the model.

In [16]:
# This cell is for CODE (numbers, a query, a check).

con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
LIMIT 10
""").df()


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>
5,client_73cda7b4e4f265ea,content_36c36abc7650d7af,2026-03-01,239,1,7.347280,<NA>
6,client_73cda7b4e4f265ea,content_a7da352b73b02668,2026-03-01,191,0,7.832461,<NA>
7,client_73cda7b4e4f265ea,content_05434271b257bb68,2026-03-01,55,0,3.272727,<NA>
8,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2026-03-01,77,0,5.636364,<NA>
9,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2026-03-01,2,0,4.500000,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [20]:
#query1
con.sql(f"""
SELECT
COUNT(*) AS total_rows,
MIN(report_date) AS first_date,
MAX(report_date) AS last_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [21]:
#query2 — Availability
con.sql(f"""
SELECT
COUNT(*) AS rows_with_ga4
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
AND ga4_data_available IS TRUE
""").df()

,rows_with_ga4
0,413966


In [22]:
#query 3 — Grain check

con.sql(f"""
SELECT
client_hash_id,
content_hash_id,
report_date,
COUNT(*) AS cnt
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY
client_hash_id,
content_hash_id,
report_date
HAVING COUNT(*)>1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,cnt


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
### Data limitations

#- Different clients have different amounts of historical data.
#- Early rows may only contain Search Console data.
#- GA4 data is unavailable for some rows.
#- This dataset cannot explain why rankings changed; it only records observed performance.

In [19]:
# This cell is for CODE (numbers, a query, a check).
con.sql(f"""
SELECT
MIN(report_date),
MAX(report_date)
FROM {TABLES['fact_daily']}
""").df()

# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min(report_date),max(report_date)
0,2025-01-27,2026-06-30


- [x] Every section above is filled
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words
- [x] Committed to my repo under `work/notebooks/`